# Hello MLflow Tracking

In [1]:
# Prereqquisites
from pyspark.sql import SparkSession

In [2]:
# Spark Session and Context
spark = SparkSession.builder.master("local") \
        .appName("Hello MLFlow") \
        .config("spark.jars.packages", "org.mlflow:mlflow-spark:2.5.0") \
        .getOrCreate()
print("Spark Version: ", spark.version)

Spark Version:  3.4.1


### Tracking RandomForest Classifier with MLflow

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

file_path = "data/sf-airbnb/sf-airbnb-clean.parquet"

df_airbnb = spark.read.parquet(file_path)
(df_train, df_test) = df_airbnb.randomSplit([0.8, 0.2], seed=42)
print("df_train # of rows: ", df_train.count())
print("df_test # of rows: ", df_test.count())


df_train # of rows:  5780
df_test # of rows:  1366


In [12]:
categorical_cols = [field for (field, data_type) in df_train.dtypes if data_type == "string"]
print("Categorical Columns: ", categorical_cols)

index_output_cols = [x + "Index" for x in categorical_cols]
print("Index Output Columns: ", index_output_cols)

string_indexer = StringIndexer(inputCols=categorical_cols,
                               outputCols=index_output_cols,
                               handleInvalid="skip")

numeric_cols = [field for (field, data_type) in df_train.dtypes if ((data_type == "double") & (field != "price"))]
print("Numeric Columns: ", numeric_cols)

assembler_input = index_output_cols + numeric_cols

vec_assembler = VectorAssembler(inputCols=assembler_input,
                                outputCol="features")

rf = RandomForestRegressor(labelCol="price", maxBins=40, maxDepth=5, numTrees=100, seed=42)

pipeline = Pipeline(stages=[string_indexer, vec_assembler, rf])


Categorical Columns:  ['host_is_superhost', 'cancellation_policy', 'instant_bookable', 'neighbourhood_cleansed', 'property_type', 'room_type', 'bed_type']
Index Output Columns:  ['host_is_superhostIndex', 'cancellation_policyIndex', 'instant_bookableIndex', 'neighbourhood_cleansedIndex', 'property_typeIndex', 'room_typeIndex', 'bed_typeIndex']
Numeric Columns:  ['host_total_listings_count', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'number_of_reviews', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'bedrooms_na', 'bathrooms_na', 'beds_na', 'review_scores_rating_na', 'review_scores_accuracy_na', 'review_scores_cleanliness_na', 'review_scores_checkin_na', 'review_scores_communication_na', 'review_scores_location_na', 'review_scores_value_na']


### MLflow

In [17]:
import mlflow
import mlflow.spark
import pandas as pd

mlflow.set_tracking_uri('http://localhost:5000')

with mlflow.start_run(run_name="random_forest") as my_run:
    mlflow.set_tag("Created by", "Jari Honkanen")
    mlflow.set_tag("Description", "SEAS8515 Project 3")

    # Log NumTrees and MaxDepth
    print("num_trees", rf.getNumTrees())
    print("maax_depth", rf.getMaxDepth())
    mlflow.log_param("num_trees", rf.getNumTrees())
    mlflow.log_param("maax_depth", rf.getMaxDepth())

    # Log model
    pipeline_model = pipeline.fit(df_train)
    mlflow.spark.log_model(pipeline_model, "model")

    # Log metrics
    df_pred = pipeline_model.transform(df_test)
    regression_eval = RegressionEvaluator(predictionCol="prediction", labelCol="price")
    rmse = regression_eval.setMetricName("rmse").evaluate(df_pred)
    r2 = regression_eval.setMetricName("r2").evaluate(df_pred)
    print(f"rmse: {rmse}, r2: {r2}")
    mlflow.log_metrics({"rmse": rmse, "r2": r2})

    # Log Artifact: Feature importance score
    rf_model = pipeline_model.stages[-1]
    df_pandas = (pd.DataFrame(list(zip(vec_assembler.getInputCols(),
                                       rf_model.featureImportances)),
                                       columns=["feature", "importance"])
                    .sort_values(by="importance", ascending=False))
    
    # Write the local file system first, then tell MLflow where to find the file
    df_pandas.to_csv("/tmp/feature-importance.csv")
    mlflow.log_artifact("/tmp/feature-importance.csv")



num_trees 100
maax_depth 5


2025/01/23 00:09:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


rmse: 211.5096898777315, r2: 0.22794251914574226
🏃 View run random_forest at: http://localhost:5000/#/experiments/0/runs/821955dab0ef49cf87495dbc5f59f3af
🧪 View experiment at: http://localhost:5000/#/experiments/0


### Using MLflow client (REST API)

In [19]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
runs = client.search_runs(my_run.info.experiment_id,
                          order_by=["attributes.start_time desc"],
                          max_results = 1)
run_id = runs[0].info.run_id
runs[0].data.metrics

{'rmse': 211.5096898777315, 'r2': 0.22794251914574226}